# IT-Type Job Classification

Dataset: https://www.kaggle.com/datasets/udacity/armenian-online-job-postings?resource=download

In [ ]:
# Import all libraries
import os
import random
import re
from pathlib import Path

import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

import nltk
from nltk import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import wordpunct_tokenize

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    Embedding,
    TextVectorization,
)
from tensorflow.keras.models import Sequential

In [ ]:
# Configuration and reproducibility
SEED = 42
MAX_ROWS = 12_000
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
MAX_TFIDF_FEATURES = 40_000
MAX_SEQUENCE_TOKENS = 30_000
SEQUENCE_LENGTH = 400
BATCH_SIZE = 64
EPOCHS = 3

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

for resource in ("stopwords", "wordnet", "omw-1.4"):
    nltk.download(resource, quiet=True)

## Reusable pipeline functions

In [ ]:
# Define reusable functions for data loading, text preprocessing, splitting, and evaluation.
def resolve_data_path(filename="online-job-postings.csv"):
    """Find the dataset whether the kernel starts in the project root or impl/."""
    candidates = [Path.cwd() / "data" / filename, Path.cwd().parent / "data" / filename]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(f"Could not find {filename}. Checked: {candidates}")


def load_data(path, max_rows=MAX_ROWS, seed=SEED):
    """Load, validate, clean labels, and take a random reproducible sample."""
    frame = pd.read_csv(path)
    required = {"jobpost", "IT"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    frame = frame[["jobpost", "IT"]].dropna().copy()
    label_map = {True: 1, False: 0, "TRUE": 1, "FALSE": 0, 1: 1, 0: 0}
    frame["IT"] = frame["IT"].map(label_map)
    if frame["IT"].isna().any():
        raise ValueError("The IT column contains unsupported labels.")

    frame["IT"] = frame["IT"].astype("int8")

    if max_rows and len(frame) > max_rows:
        frame, _ = train_test_split(
            frame, train_size=max_rows, stratify=frame["IT"], random_state=seed
        )

    return frame.sample(frac=1, random_state=seed).reset_index(drop=True)


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"&[a-z]+;", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()


def preprocess_text(text):
    """Clean, tokenize, remove stop words, and lemmatize one document."""
    tokens = (token for token in wordpunct_tokenize(clean_text(text)) if len(token) > 1)
    tokens = (token for token in tokens if token not in STOP_WORDS)
    return " ".join(LEMMATIZER.lemmatize(token, pos="v") for token in tokens)


def split_data(frame):
    train_val, test = train_test_split(
        frame, test_size=TEST_SIZE, stratify=frame["IT"], random_state=SEED
    )

    train, validation = train_test_split(
        train_val,
        test_size=VALIDATION_SIZE,
        stratify=train_val["IT"],
        random_state=SEED,
    )

    return train, validation, test


def evaluate_model(name, y_true, predictions):
    print(f"\n{name}\n{'=' * len(name)}")
    print(classification_report(y_true, predictions, digits=4))
    matrix_display = ConfusionMatrixDisplay.from_predictions(
        y_true, predictions, cmap="Blues"
    )

    plt.title(f"{name} confusion matrix")
    result_dir = data_path.parent.parent / "result"
    result_dir.mkdir(parents=True, exist_ok=True)

    plot_path = result_dir / "best_model_confusion_matrix.png"
    matrix_display.figure_.savefig(plot_path, dpi=300, bbox_inches="tight")
    plt.close(matrix_display.figure_)

    print(f"Saved confusion matrix to {plot_path}")

## Load and prepare data

In [ ]:
# Load, preprocess, and preview the sampled job-posting dataset.
data_path = resolve_data_path()
df = load_data(data_path)
df["clean_text"] = df["jobpost"].map(preprocess_text)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

print(f"Loaded {len(df):,} rows from {data_path}")
display(df["IT"].value_counts().rename(index={0: "Non-IT", 1: "IT"}).to_frame("rows"))
display(df[["jobpost", "clean_text", "IT"]].head())

In [ ]:
# Split the data and extract text and label arrays for each subset.
train_df, validation_df, test_df = split_data(df)

x_train_text = train_df["clean_text"].to_numpy()
x_validation_text = validation_df["clean_text"].to_numpy()
x_test_text = test_df["clean_text"].to_numpy()
y_train = train_df["IT"].to_numpy()
y_validation = validation_df["IT"].to_numpy()
y_test = test_df["IT"].to_numpy()

print(
    f"Train: {len(train_df):,} | Validation: {len(validation_df):,} | Test: {len(test_df):,}"
)

## TF-IDF models

In [ ]:
# Fit TF-IDF on training text and transform all data subsets.
vectorizer = TfidfVectorizer(
    max_features=MAX_TFIDF_FEATURES,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
)

x_train_tfidf = vectorizer.fit_transform(x_train_text)
x_validation_tfidf = vectorizer.transform(x_validation_text)
x_test_tfidf = vectorizer.transform(x_test_text)
print(x_train_tfidf.shape, x_validation_tfidf.shape, x_test_tfidf.shape)

In [ ]:
# A linear SVM is well suited to sparse, high-dimensional text features.
svm_model = LinearSVC(C=1.0, class_weight="balanced", random_state=SEED)
svm_model.fit(x_train_tfidf, y_train)

In [ ]:
# Score the trained SVM on the validation set.
svm_validation_predictions = svm_model.predict(x_validation_tfidf)
svm_validation_f1 = f1_score(y_validation, svm_validation_predictions, average="macro")

print(f"Linear SVM validation macro-F1: {svm_validation_f1:.4f}")

In [ ]:
# Random Forest baseline. Sparse input avoids the former dense conversion.
rf_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=6,
    random_state=SEED,
)

rf_model.fit(x_train_tfidf, y_train)

In [ ]:
# Score the trained Random Forest on the validation set.
rf_validation_predictions = rf_model.predict(x_validation_tfidf)
rf_validation_f1 = f1_score(y_validation, rf_validation_predictions, average="macro")

print(f"Random Forest validation macro-F1: {rf_validation_f1:.4f}")

## Sequence Bi-LSTM

Unlike the former one-timestep TF-IDF input, this model receives ordered token sequences and can learn word-order patterns. The validation set—not the test set—controls early stopping.

In [ ]:
# Build and compile the sequence-based Bi-LSTM classifier.
text_vectorizer = TextVectorization(
    max_tokens=MAX_SEQUENCE_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
)

text_vectorizer.adapt(x_train_text)

In [ ]:
# Create Bi-LSTM Model
bilstm_model = Sequential(
    [
        tf.keras.Input(shape=(), dtype=tf.string),
        text_vectorizer,
        Embedding(MAX_SEQUENCE_TOKENS, 128, mask_zero=True),
        Bidirectional(LSTM(64)),
        Dropout(0.4),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid"),
    ]
)

bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)

In [ ]:
# Display the Bi-LSTM architecture and parameter counts.
bilstm_model.summary()

In [ ]:
# Compute class weights and train the Bi-LSTM with validation callbacks.
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-5),
]

history = bilstm_model.fit(
    x_train_text,
    y_train,
    validation_data=(x_validation_text, y_validation),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=callbacks,
)

In [ ]:
# Save the Bi-LSTM training history as an image under result/.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="validation")
axes[0].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="validation")
axes[1].set(title="Loss", xlabel="Epoch", ylabel="Loss")

for axis in axes:
    axis.legend()
    axis.grid(alpha=0.2)

result_dir = data_path.parent.parent / "result"
result_dir.mkdir(parents=True, exist_ok=True)
history_plot_path = result_dir / "bilstm_training_history.png"
plt.tight_layout()
fig.savefig(history_plot_path, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved training plot to {history_plot_path}")

In [ ]:
# Predict validation labels and compute the Bi-LSTM macro-F1 score.
bilstm_validation_probabilities = bilstm_model.predict(
    x_validation_text, batch_size=BATCH_SIZE
).ravel()

bilstm_validation_predictions = (bilstm_validation_probabilities >= 0.5).astype("int8")

bilstm_validation_f1 = f1_score(
    y_validation, bilstm_validation_predictions, average="macro"
)

print(f"Bi-LSTM validation macro-F1: {bilstm_validation_f1:.4f}")

In [ ]:
# Select by validation macro-F1, evaluate once on the untouched test set, and save.
validation_scores = {
    "linear_svm": svm_validation_f1,
    "random_forest": rf_validation_f1,
    "bilstm": bilstm_validation_f1,
}

best_model_name = max(validation_scores, key=validation_scores.get)

project_root = data_path.parent.parent
model_dir = project_root / "result" / "model"
model_dir.mkdir(parents=True, exist_ok=True)

if best_model_name == "linear_svm":
    best_predictions = svm_model.predict(x_test_tfidf)
    model_path = model_dir / "best_model.joblib"
    joblib.dump(
        {"name": best_model_name, "model": svm_model, "vectorizer": vectorizer},
        model_path,
    )
elif best_model_name == "random_forest":
    best_predictions = rf_model.predict(x_test_tfidf)
    model_path = model_dir / "best_model.joblib"
    joblib.dump(
        {"name": best_model_name, "model": rf_model, "vectorizer": vectorizer},
        model_path,
    )
else:
    best_probabilities = bilstm_model.predict(
        x_test_text, batch_size=BATCH_SIZE
    ).ravel()
    best_predictions = (best_probabilities >= 0.5).astype("int8")
    model_path = model_dir / "best_model.keras"
    bilstm_model.save(model_path)

In [ ]:
# Report validation rankings, saved model path, and final test performance.
print("Validation macro-F1 scores:")
display(pd.Series(validation_scores, name="macro_f1").sort_values(ascending=False))
print(f"Saved {best_model_name} to {model_path}")
evaluate_model(f"Best model: {best_model_name}", y_test, best_predictions)

In [ ]:
# Release TensorFlow resources after training and evaluation.
tf.keras.backend.clear_session()